# Module 4.2: Isolation Forests & Elliptic Envelope for Zero-Label Anomalies
### 🇵🇱 Lasy Izolacyjne w Bezetykietowej Detekcji Anomalii Maszyn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m04_l02_isolation_forests_unlabeled_anomalies.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Train tree-based Isolation Forests and Elliptic Envelopes on raw multi-sensor telemetry to assign continuous anomaly health scores without historical failure logs.  
> 🇵🇱 **Streszczenie:** Trenuj Lasy Izolacyjne na danych z czujników, wyliczając ciągły wskaźnik zdrowia maszyny bez konieczności posiadania historii uszkodzeń.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Unlabeled Factory Anomaly Detection**: Plant breakdowns occur rarely. Supervised models fail due to severe lack of failure labels.
2. **Isolation Forest Geometry**: Anomalous samples require far fewer random splits to isolate ($h(x) \ll c(n)$) compared to dense normal operational points.
3. **Continuous Anomaly Scoring**:
   $$s(x, n) = 2^{-\frac{\mathbb{E}(h(x))}{c(n)}}$$
   Scoring continuous telemetry allows early warning alerting as the score climbs from 0.35 towards 0.70.


In [ ]:
# 1. Multi-Sensor Telemetry with Incipient Mechanical Binding
from sklearn.ensemble import IsolationForest

np.random.seed(42)
n_samples = 1500

# Normal operational telemetry: Motor Current, Bearing Temp, Vibration
X_normal = np.column_stack([
    np.random.normal(30.0, 3.0, n_samples),
    np.random.normal(65.0, 4.0, n_samples),
    np.random.normal(1.8, 0.3, n_samples)
])

# Inject subtle multi-sensor binding fault
X_fault = np.column_stack([
    np.random.uniform(42.0, 55.0, 25),
    np.random.uniform(78.0, 92.0, 25),
    np.random.uniform(2.8, 4.2, 25)
])

X_all = np.vstack([X_normal, X_fault])
iso_forest = IsolationForest(contamination=0.02, random_state=42)
iso_forest.fit(X_all)
anomaly_scores = -iso_forest.score_samples(X_all)  # Higher = more anomalous


In [ ]:
# 2. Continuous Anomaly Timeline Inspection
plt.figure(figsize=(14, 5))
plt.plot(anomaly_scores, color='#0284c7', alpha=0.7, label='Isolation Forest Anomaly Score')
plt.axhline(0.60, color='#ef4444', linestyle='--', linewidth=2, label='Safety Trip Alarm Threshold (0.60)')
plt.axvspan(n_samples, len(X_all), color='#ef4444', alpha=0.2, label='Injected Mechanical Binding Zone')

plt.title("Continuous Unsupervised Telemetry Anomaly Scoring", fontsize=13, fontweight='bold')
plt.xlabel("Sample Timeline Index")
plt.ylabel("Anomaly Score [0 to 1]")
plt.legend(loc='upper left')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Fit sklearn.ensemble.IsolationForest on healthy baseline machine telemetry *(🇵🇱 Dopasuj IsolationForest do wzorcowych danych ze sprawnych cykli produkcyjnych)*
- [ ] **Task 2:** Compute decision_function continuous anomaly health scores across operating shifts *(🇵🇱 Oblicz ciągły wskaźnik anomalii za pomocą decision_function na przestrzeni zmian roboczych)*
- [ ] **Task 3:** Set dynamic alarm thresholds based on statistical anomaly score percentiles *(🇵🇱 Skonfiguruj dynamiczne progi alarmowe w oparciu o percentyle wskaźnika anomalii)*


In [ ]:
# Exercise Challenge:
# Verify that average anomaly score in the fault zone is significantly higher than baseline
baseline_score = np.mean(anomaly_scores[:n_samples])
fault_score = np.mean(anomaly_scores[n_samples:])

print(f"Baseline Anomaly Score : {baseline_score:.3f}")
print(f"Fault Zone Anomaly Score: {fault_score:.3f}")
assert fault_score > baseline_score * 1.5, "Anomaly score failed to reflect mechanical binding."
print("✓ Isolation Forest anomaly verification passed.")
